<a href="https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = one content page's daily performance for one client, identified by `client_hash_id`, `content_hash_id`, and `report_date` in `fact_content_daily_performance`.

**Time window:** I will use March 2026 as the development month. The first half of March will provide the information available for ranking, while the second half will be used only to construct the observed decline proxy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

*   **feature:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`,
`sessions_organic`, `sessions_ai` from the first half of March.


* **label / proxy:** `is_declining`, based on the observed change in `gsc_impressions` between the first and second half of March.

* **context:** `client_hash_id`, `content_hash_id`, `report_date`, `month`

* **excluded:**

  * Second half `gsc_impressions` because it is used to construct the decline proxy.
  *   `content_hash_id` and `client_hash_id` because they are identifiers, not meaningful predictive signals.

  *  `report_date` because the date itself should not determine refresh priority.

  * Future period performance because it would not be available when making the ranking
  *  Any decline or trend derived field because it would directly expose the outcome we are trying to evaluate.
  * Rows where `gsc_data_available IS NOT TRUE` because unavailable GSC data should not be treated as zero performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
# --- Setup: connect DuckDB to the Hugging Face warehouse ---

import os
import duckdb
import pandas as pd
from google.colab import userdata

# Read the HF token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.sql("INSTALL httpfs; LOAD httpfs;")

# Register the token as a DuckDB secret
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
)
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

FACT_TABLE_GLOB = (
    f"{WAREHOUSE}/fact_content_daily_performance/**/*.parquet"
)

print("DuckDB connected and Hugging Face secret registered.")

DuckDB connected and Hugging Face secret registered.


In [5]:
march_path = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

print(march_path)

schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
""").df()

display(schema)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


### Grain check

I expect the warehouse grain to be one content page for one client on one report date.
I will check that there are no duplicate combinations of `client_hash_id`, `content_hash_id`, and `report_date`.

In [6]:
grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM read_parquet('{march_path}')
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Duplicate combinations found: {len(grain_check)}")
display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate combinations found: 0


,client_hash_id,content_hash_id,report_date,row_count


### March 2026 coverage

Veryfying the number of rows in the March 2026 slice and confirm the actual date range covered by the data.

In [7]:
march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{march_path}')
""").df()

display(march_summary)

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### GSC availability

I will keep only rows where GSC data is explicitly available.
This prevents unavailable search data from being interpreted as zero performance.

In [8]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM read_parquet('{march_path}')
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows
0,9841378,3611061


### Features

The decision moment is the first 15 days of March 2026. Only information available during this period can be used as a feature.

The goal is to describe each page's existing visibility, search performance, organic traffic, and engagement before deciding which pages deserve review.

I use five signals: GSC impressions, GSC average position, GSC CTR, organic sessions, and GA4 engaged sessions.

These features describe different aspects of the page's existing performance without using information from the later outcome window.

In [18]:
df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        sessions_organic,
        ga4_engaged_sessions,
        gsc_data_available
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
""").df()

df["report_date"] = pd.to_datetime(df["report_date"])

print(f"Total rows loaded: {len(df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows loaded: 3,611,061


### Decision and outcome windows

I split March into two windows to make the timing of the analysis explicit.

Days 1 through 15 are the decision window and are used to build the features.

Days 16 through 31 are the outcome window and are kept separate so that future information cannot enter the features.

In [19]:
early = df[df["report_date"].dt.day <= 15].copy()
late = df[df["report_date"].dt.day > 15].copy()

print(f"Decision window rows: {len(early):,}")
print(f"Outcome window rows: {len(late):,}")

Decision window rows: 1,640,237
Outcome window rows: 1,970,824


### Building the feature frame

Each feature is calculated only from days 1 through 15.

GSC impressions measure how much search visibility the page already receives.

GSC average position gives context for that visibility by showing where the page appears in search.

GSC CTR measures how effectively existing impressions turn into clicks and is calculated only from early window clicks and impressions.

Organic sessions provide a second view of traffic reaching the page.

GA4 engaged sessions provide an engagement signal so the ranking is not based only on search visibility.

In [20]:
feature_frame = (
    early
    .groupby(["client_hash_id", "content_hash_id"])
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        sessions_organic=("sessions_organic", "sum"),
        ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
        gsc_clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

feature_frame["gsc_ctr"] = (
    feature_frame["gsc_clicks"]
    / feature_frame["gsc_impressions"].replace(0, pd.NA)
)

feature_frame = feature_frame.drop(columns=["gsc_clicks"])

display(feature_frame.head())
print("Rows:", len(feature_frame))

,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,sessions_organic,ga4_engaged_sessions,gsc_ctr
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,12.639599,0,0,0.008403
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,9.225529,0,0,0.000000
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,8.094074,0,0,0.000000
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,12.587226,0,0,0.000000
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,11.500000,0,0,0.000000


Rows: 151981


### Feature reasoning

`gsc_impressions` tells us whether a page has meaningful existing search visibility.

`gsc_avg_position` tells us how well that visible page is positioned in search.

`gsc_ctr` tells us how much of that available visibility becomes clicks.

`sessions_organic` gives us an additional measure of organic traffic reaching the page.

`ga4_engaged_sessions` adds an engagement signal so the ranking considers what happens after traffic reaches the page.

All five are calculated from the decision window only, so none uses the future outcome period.

### Feature timing check

The feature frame is created entirely from the first 15 days of March.

The latest date used for every feature is therefore March 15.

The second half of March is intentionally excluded from feature construction and will only be used later as the observed outcome window.

In [21]:
print("Feature window starts:", early["report_date"].min().date())
print("Feature window ends:", early["report_date"].max().date())

Feature window starts: 2026-03-01
Feature window ends: 2026-03-15


### Future outcome proxy

The second half of March is used only to create an observed decline proxy.

For each page, I compare its average GSC impressions in the first and second halves of the month.

A page is marked as declining when its second half average is at least 20 percent lower than its first half average.

This is only an observed evaluation proxy. It does not mean that refreshing the page would cause its performance to improve.

In [24]:
outcome_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        AVG(gsc_impressions) FILTER (
            WHERE report_date <= DATE '2026-03-15'
        ) AS first_half_impressions,

        AVG(gsc_impressions) FILTER (
            WHERE report_date > DATE '2026-03-15'
        ) AS second_half_impressions

    FROM read_parquet('{march_path}')

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING
        COUNT(*) FILTER (
            WHERE report_date <= DATE '2026-03-15'
        ) > 0
        AND
        COUNT(*) FILTER (
            WHERE report_date > DATE '2026-03-15'
        ) > 0
""").df()

outcome_frame["is_declining"] = (
    outcome_frame["second_half_impressions"]
    <= outcome_frame["first_half_impressions"] * 0.80
)

display(outcome_frame.head())

,client_hash_id,content_hash_id,first_half_impressions,second_half_impressions,is_declining
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,28.600000,44.4375,False
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1.636364,2.6000,False
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,5.933333,4.0000,True
3,client_73cda7b4e4f265ea,content_05434271b257bb68,41.866667,49.5625,False
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,85.333333,93.1250,False


### Deliberate leakage experiment

I will now intentionally introduce the future outcome into the feature set.

The `is_declining` value is calculated using the later part of March, so it would not be available at the decision moment.

If I use it to rank pages, the ranking should look unrealistically good because the feature contains the outcome itself.

This is deliberately wrong and is included only to demonstrate leakage.

In [25]:
leaked_frame = feature_frame.merge(
    outcome_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Deliberately leaked score
leaked_frame["leaked_score"] = leaked_frame["is_declining"].astype(int)

leaked_frame = leaked_frame.sort_values(
    "leaked_score",
    ascending=False
)

k = min(100, len(leaked_frame))

leaked_precision_at_k = (
    leaked_frame.head(k)["is_declining"].mean()
)

print(f"Deliberately leaked Precision@{k}: {leaked_precision_at_k:.3f}")

Deliberately leaked Precision@100: 1.000


### Removing the leaked information

The leakage experiment used `is_declining`, which was calculated from the future outcome window.

That information would not exist when the editor makes the refresh prioritization decision.

I therefore remove it from the feature set and keep the five original features as the honest input.

In [26]:
honest_features = feature_frame.copy()

print("Honest feature columns:")
print([
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_ctr",
    "sessions_organic",
    "ga4_engaged_sessions"
])

print("\nOutcome column kept separately:")
print("is_declining")

Honest feature columns:
['gsc_impressions', 'gsc_avg_position', 'gsc_ctr', 'sessions_organic', 'ga4_engaged_sessions']

Outcome column kept separately:
is_declining


### Final leakage check

The final feature frame contains only information from March 1 through March 15.

The decline proxy is stored separately and is based on the later outcome window.

Therefore the future outcome is not being used as an input feature.

In [27]:
forbidden_columns = {
    "is_declining",
    "first_half_impressions",
    "second_half_impressions"
}

leaked_columns = forbidden_columns.intersection(
    honest_features.columns
)

print("Leaked columns in final feature frame:", leaked_columns)

assert len(leaked_columns) == 0

print("Leakage check passed.")

Leaked columns in final feature frame: set()
Leakage check passed.


## 4. Data limits

* A key limitation is that the observed decline proxy is based only on the first and second halves of March 2026.

* This is a relatively short observation window, so a page may appear to decline because of normal short term variation rather than a meaningful content problem.

* The five features describe observed page performance using `gsc_impressions`, `gsc_avg_position`, `gsc_ctr`, `sessions_organic`, and `ga4_engaged_sessions`, but they cannot tell us whether refreshing a page would actually improve its future performance.

Therefore, the resulting ranking should be treated as decision support for deciding which pages deserve review, not as proof that a page needs a refresh or that a refresh will cause recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- Every section above is filled — markdown thinking AND the code that backs it
- The notebook runs top to bottom with no errors (Runtime → Run all)
- No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
- Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.